In [1]:
import sys
print(sys.executable)

C:\FIAP\ML\medical-triage-mlops\.venv\Scripts\python.exe


In [2]:
import pandas as pd
from pathlib import Path

In [3]:
DATA_DIR = Path("../data/raw")

train_df = pd.read_csv(DATA_DIR / "medical_tc_train.csv")
test_df = pd.read_csv(DATA_DIR / "medical_tc_test.csv")
labels_df = pd.read_csv(DATA_DIR / "medical_tc_labels.csv")

In [4]:
print(train_df.shape)
print(test_df.shape)
print(labels_df.shape)

(11550, 2)
(2888, 2)
(5, 2)


In [5]:
labels_df

,condition_label,condition_name
0,1,neoplasms
1,2,digestive system diseases
2,3,nervous system diseases
3,4,cardiovascular diseases
4,5,general pathological conditions


In [6]:
train_df.head()

,condition_label,medical_abstract
0,5,Tissue changes around loose prostheses. A cani...
1,1,Neuropeptide Y and neuron-specific enolase lev...
2,2,"Sexually transmitted diseases of the colon, re..."
3,1,Lipolytic factors associated with murine and h...
4,3,Does carotid restenosis predict an increased r...


In [7]:
test_df.head()

,condition_label,medical_abstract
0,3,Obstructive sleep apnea following topical orop...
1,5,Neutrophil function and pyogenic infections in...
2,5,A phase II study of combined methotrexate and ...
3,1,Flow cytometric DNA analysis of parathyroid tu...
4,4,Paraneoplastic vasculitic neuropathy: a treata...


In [8]:
train_df["condition_label"].value_counts()

condition_label
5    3844
1    2530
4    2441
3    1540
2    1195
Name: count, dtype: int64

In [9]:
test_df["condition_label"].value_counts()

condition_label
5    961
1    633
4    610
3    385
2    299
Name: count, dtype: int64

In [10]:
train_df["condition_label"].value_counts(normalize=True).sort_index()

condition_label
1    0.219048
2    0.103463
3    0.133333
4    0.211342
5    0.332814
Name: proportion, dtype: float64

In [11]:
test_df["condition_label"].value_counts(normalize=True).sort_index()

condition_label
1    0.219183
2    0.103532
3    0.133310
4    0.211219
5    0.332756
Name: proportion, dtype: float64

In [12]:
print(train_df["medical_abstract"].iloc[0])

Tissue changes around loose prostheses. A canine model to investigate the effects of an antiinflammatory agent. The aseptically loosened prosthesis provided a means for investigating the in vivo and in vitro activity of the cells associated with the loosening process in seven dogs. The cells were isolated and maintained in culture for sufficient periods of time so that their biologic activity could be studied as well as the effect of different agents added to the cells in vivo or in vitro. The biologic response as determined by interleukin-1 and prostaglandin E2 activity paralleled the roentgenographic appearance of loosening and the technetium images and observations made at the time of revision surgery. The correlation between clinical, roentgenographic, histologic, and biochemical loosening indicates that the canine model is suitable for investigating the mechanisms of prosthetic failure. A canine model permits the study of possible nonsurgical therapeutic interventions with the ult

In [13]:
len(train_df["medical_abstract"].iloc[0].split())

156

In [14]:
train_df["word_count"] = (
    train_df["medical_abstract"]
    .fillna("")
    .str.split()
    .str.len()
)

train_df["word_count"].describe()

count    11550.000000
mean       179.640606
std         76.369776
min         24.000000
25%        122.000000
50%        175.000000
75%        234.000000
max        596.000000
Name: word_count, dtype: float64

In [15]:
train_df["medical_abstract"].duplicated().sum()

np.int64(2105)

In [16]:
train_texts = set(train_df["medical_abstract"].dropna())
test_texts = set(test_df["medical_abstract"].dropna())

overlap = train_texts.intersection(test_texts)

len(overlap)

988

In [17]:
list(overlap)[:5]

['Risk of gastric cancer after gastric surgery for benign disorders. The objective of this review was to evaluate published evidence for the association between gastric resection for benign disorders and subsequent cancer of the gastric remnant. We searched the literature through Medline (1970 to 1988) and through the references of relevant articles. Fifty-eight studies consisting of case series, uncontrolled surveys, and case control or cohort analyses were identified and critically assessed using defined methodological criteria. There were no consistent differences between the expected and observed number of cancers occurring within 15 years after gastric resection. However, all case control studies and seven of the eight cohort analyses, in which the prevalence of cancer was stratified by time since gastric resection, indicated a twofold to fourfold increase in the risk of gastric cancer in patients who survived 15 or more years after gastric surgery. We conclude that most studies o

In [18]:
grupos = train_df.groupby("medical_abstract")["condition_label"].agg(["size", "nunique"])
duplicados = grupos[grupos["size"] > 1]

print(len(duplicados))
print((duplicados["nunique"] > 1).sum())

1956
1956


In [19]:
# 1) O teste também tem textos repetidos com rótulos conflitantes?
grupos_teste = test_df.groupby("medical_abstract")["condition_label"].agg(["size", "nunique"])
print((grupos_teste["size"] > 1).sum(), (grupos_teste["nunique"] > 1).sum())

# 2) Dos textos que estão no treino e no teste, quantos têm rótulos diferentes entre os dois lados?
rot_treino = train_df.groupby("medical_abstract")["condition_label"].agg(set)
rot_teste = test_df.groupby("medical_abstract")["condition_label"].agg(set)
comuns = rot_treino.index.intersection(rot_teste.index)
diferentes = sum(rot_treino[t] != rot_teste[t] for t in comuns)
print(len(comuns), diferentes)

113 113
988 988


In [20]:
cols = ["medical_abstract", "condition_label"]
all_df = pd.concat(
    [train_df[cols].assign(split="train"), test_df[cols].assign(split="test")],
    ignore_index=True,
)

n_rotulos = all_df.groupby("medical_abstract")["condition_label"].transform("nunique")
limpo = all_df[n_rotulos == 1]

train_clean = limpo[limpo["split"] == "train"].drop(columns="split")
test_clean = limpo[limpo["split"] == "test"].drop(columns="split")

print(len(train_clean), len(test_clean))
print(train_clean["condition_label"].value_counts(normalize=True).sort_index())
print(test_clean["condition_label"].value_counts(normalize=True).sort_index())

assert set(train_clean["medical_abstract"]).isdisjoint(test_clean["medical_abstract"])
assert not train_clean["medical_abstract"].duplicated().any()

6607 1691
condition_label
1    0.265173
2    0.084456
3    0.124716
4    0.235054
5    0.290601
Name: proportion, dtype: float64
condition_label
1    0.261975
2    0.083383
3    0.133057
4    0.241277
5    0.280308
Name: proportion, dtype: float64
